In [4]:
import os
import joblib
import pandas as pd
import gradio as gr

# -------------------------------------------------------------
# 1. RESOLVE PATHS AND LOAD MODEL (Jupyter Optimized)
# -------------------------------------------------------------
# Use getcwd() because __file__ is not defined inside Jupyter Notebook cells
CURRENT_DIR = os.getcwd()

# Points to the model folder one level up from your notebook's directory
MODEL_PATH = os.path.normpath(os.path.join(CURRENT_DIR, "../models/xgb.joblib"))

print(f"Attempting to load model from: {MODEL_PATH}")

try:
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully!")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find xg.joblib at '{MODEL_PATH}'. "
        "Please check your notebook's folder location!"
    )

# -------------------------------------------------------------
# 2. DEFINE PREDICTION FUNCTION WITH EXPLICIT FEATURE ORDER
# -------------------------------------------------------------
# This array matches your final 21 training columns in exact physical order
FEATURE_COLUMNS = [
    'loanamount_x', 'totaldue_x', 'termdays_x', 'longitude_gps', 'latitude_gps',
    'loanamount_y', 'totaldue_y', 'termdays_y', 'age', 'first_payment_delay_days',
    'prev_actual_days', 'is_weekend', 'daily_payment_burden',
    'employment_status_clients_Permanent', 'employment_status_clients_Retired',
    'employment_status_clients_Self-Employed', 'employment_status_clients_Student',
    'employment_status_clients_Unemployed', 'bank_account_type_Other',
    'bank_account_type_Savings', 'bank_name_encoded'
]

def predict_loan_status(
    loanamount_x, totaldue_x, termdays_x, longitude_gps, latitude_gps,
    loanamount_y, totaldue_y, termdays_y, age, first_payment_delay_days,
    prev_actual_days, is_weekend, daily_payment_burden, employment_status,
    bank_account_type, bank_name_encoded
):
    # Construct a dictionary holding all 21 raw model variables
    input_data = {
        'loanamount_x': float(loanamount_x),
        'totaldue_x': float(totaldue_x),
        'termdays_x': int(termdays_x),
        'longitude_gps': float(longitude_gps),
        'latitude_gps': float(latitude_gps),
        'loanamount_y': float(loanamount_y),
        'totaldue_y': float(totaldue_y),
        'termdays_y': float(termdays_y),
        'age': int(age),
        'first_payment_delay_days': int(first_payment_delay_days),
        'prev_actual_days': int(prev_actual_days),
        'is_weekend': int(1 if is_weekend == "Yes" else 0),
        'daily_payment_burden': float(daily_payment_burden),
        # Map selected radio buttons to One-Hot encoded fields
        'employment_status_clients_Permanent': int(1 if employment_status == "Permanent" else 0),
        'employment_status_clients_Retired': int(1 if employment_status == "Retired" else 0),
        'employment_status_clients_Self-Employed': int(1 if employment_status == "Self-Employed" else 0),
        'employment_status_clients_Student': int(1 if employment_status == "Student" else 0),
        'employment_status_clients_Unemployed': int(1 if employment_status == "Unemployed" else 0),
        # Map account type choices
        'bank_account_type_Other': int(1 if bank_account_type == "Other" else 0),
        'bank_account_type_Savings': int(1 if bank_account_type == "Savings" else 0),
        'bank_name_encoded': float(bank_name_encoded)
    }
    
    # Force creation of DataFrame with exact required training shapes/orders
    input_df = pd.DataFrame([input_data])
    
    # Reorder columns to ensure execution layout matches model expectations exactly
    input_df = input_df[FEATURE_COLUMNS]
    
    # Predict loan probabilities
    probabilities = model.predict_proba(input_df)[0]
    bad_prob, good_prob = probabilities[0], probabilities[1]
    
    # Render user-friendly status badge
    status = "🟢 GOOD LOAN (Low Risk)" if good_prob >= 0.5 else "🔴 BAD LOAN (High Risk)"
    return f"**Prediction:** {status}\n\n* Confidence Good: {good_prob:.2%}\n* Confidence Bad: {bad_prob:.2%}"

# -------------------------------------------------------------
# 3. CONSTRUCT THE APPMOBILE INTERFACE
# -------------------------------------------------------------
with gr.Blocks(title="Credit Scoring System") as demo:
    gr.Markdown("# 🏦 Credit Risk Assessment & Scoring Portal")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📦 Current Loan Data")
            loanamount_x = gr.Number(label="Loan Amount Requested (x)", value=30000)
            totaldue_x = gr.Number(label="Total Due (x)", value=35000)
            termdays_x = gr.Slider(minimum=7, maximum=90, step=1, label="Term Days (x)", value=30)
            daily_payment_burden = gr.Number(label="Daily Payment Burden", value=1166.0)
            is_weekend = gr.Radio(["No", "Yes"], label="Is Weekend Request?", value="No")
            
        with gr.Column():
            gr.Markdown("### ⏳ Historical Metrics")
            age = gr.Number(label="Client Age", value=32)
            first_payment_delay_days = gr.Number(label="First Payment Delay Days", value=0)
            prev_actual_days = gr.Number(label="Previous Actual Days Taken", value=14)
            loanamount_y = gr.Number(label="Last Historical Loan Amount (y)", value=20000)
            totaldue_y = gr.Number(label="Last Historical Total Due (y)", value=24500)
            termdays_y = gr.Number(label="Last Historical Term Days (y)", value=30)
            
        with gr.Column():
            gr.Markdown("### 🔑 Demographics & Settings")
            employment_status = gr.Radio(["Permanent", "Retired", "Self-Employed", "Student", "Unemployed"], label="Employment Status", value="Permanent")
            bank_account_type = gr.Radio(["Savings", "Other"], label="Bank Account Type", value="Savings")
            bank_name_encoded = gr.Number(label="Bank Name Encoded Value", value=0.15)
            longitude_gps = gr.Number(label="GPS Longitude", value=3.379)
            latitude_gps = gr.Number(label="GPS Latitude", value=6.524)
            
    submit_btn = gr.Button("Evaluate Application Risk", variant="primary")
    output_box = gr.Markdown(label="Risk Results")
    
    submit_btn.click(
        fn=predict_loan_status,
        inputs=[
            loanamount_x, totaldue_x, termdays_x, longitude_gps, latitude_gps,
            loanamount_y, totaldue_y, termdays_y, age, first_payment_delay_days,
            prev_actual_days, is_weekend, daily_payment_burden, employment_status,
            bank_account_type, bank_name_encoded
        ],
        outputs=output_box
    )

# Run app locally within the notebook instance
demo.launch(inline=True)


Attempting to load model from: C:\Users\blaizo\Desktop\MarkGPT\contributors\mbuh-blaise-khan\module02\loan_prediction\models\xgb.joblib
Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
